# Tree Traversal, Returned Values, and Costs


```{contents}
:local:
:depth: 2
```


```{index} A Visit and a Returned Result Are Different Events
```



```{index} A Visit and a Returned Result Are Different Events
```

## A Visit and a Returned Result Are Different Events



A traversal chooses when to visit a node relative to its children. A recursive aggregation also returns results from children to their caller. These events can happen at different times. Printing a node before recursing gives preorder; computing its subtree size cannot finish until the child sizes are known.

Use a small tree with root B, left child A and right child C. Preorder visits B, A, C; inorder visits A, B, C; postorder visits A, C, B. These names describe timing, not three different storage layouts. The same nodes and edges support all three orders.

| Moment during Count(B) | Known information | What must wait? |
| --- | --- | --- |
| Enter B | B exists | Its subtree count |
| Left call returns | Left count is 1 | Right count |
| Right call returns | Right count is 1 | Nothing needed for the sum |
| Return from B | 1 + 1 + 1 is 3 | Caller may now use 3 |

An empty subtree returns zero directly. That is not an arbitrary special case appended to the algorithm; it makes the recurrence Count(node) = 1 + Count(left) + Count(right) valid for a leaf. A missing return, or adding one for an empty subtree, changes the contract across every ancestor that uses the result.

For related explanation, see the further reading below.[^development-reading]


```{index} Count Active Calls Separately from Visited Nodes
```



```{index} Count Active Calls Separately from Visited Nodes
```

## Count Active Calls Separately from Visited Nodes



A traversal visits n nodes, but it does not keep all n calls active simultaneously in every shape. On a balanced tree, the active recursive path follows the tree's height. On a chain, the height can grow with n, so the same recursion can need linear active stack depth. Total calls and maximum simultaneous calls answer different cost questions.

A queue-based level-order traversal has a different live frontier. It can hold nodes across a wide level. Choosing recursion or a queue is therefore a question about visit policy and storage as well as syntax. Neither approach makes a cyclic shared graph into a proper tree; the ownership and acyclicity assumptions from the structural lesson matter.

Guided trace: add D as C's right child. Predict all three visit orders and every returned Count value. Draw arrows for calls going down and results coming back. Then turn the four nodes into a right-only chain and explain which values remain the same and which storage claim changes. The number of nodes is still four; the maximum active depth is different.


```{index} Information Flow in Recursive Methods
```

### Information Flow in Recursive Methods

Every recursive tree method moves information in at most two directions, and naming the direction before writing the method prevents most mistakes.

| Flow | What each call receives | What each call returns | Example |
| --- | --- | --- | --- |
| Local | Only the current node | Nothing | Print every label; add one to every value |
| Passing down | The node plus context from its ancestors | Nothing | Print each node indented by its depth; mark each node with its department path |
| Collect and return | Only the current node | A summary of its subtree | `Count`, `Sum`, `Height`, `CountLeaves` |
| Both | Context from above | A summary for below | Is there a root-to-leaf path whose values total a target? |

Passing down means adding a parameter that each call adjusts before recursing: a depth that grows by one, a running path, a bound inherited from an ancestor as in the BST validator of the next section. Collecting and returning means the method has a return type and every recursive call's result is captured and combined. The two classic defects are a recursive call whose result is thrown away, so the summary is wrong, and a passed-down value that is never adjusted, so every node sees the root's context. The example shows one method of each kind on a department tree whose values are budgets.

In [ ]:
using System;

var company = new Dept("Company", 0,
    new Dept("Sales", 120, new Dept("Inside", 40), new Dept("Field", 65)),
    new Dept("Operations", 90, new Dept("Warehouse", 30), null));

PrintIndented(company, 0);                       // passing down: depth
Console.WriteLine($"total budget = {Total(company)}");           // collect and return
Console.WriteLine($"path totaling 225? {HasPathTotal(company, 225)}");   // both directions
Console.WriteLine($"path totaling 200? {HasPathTotal(company, 200)}");

static void PrintIndented(Dept? d, int depth)
{
    if (d is null) return;
    Console.WriteLine($"{new string(' ', depth * 2)}{d.Name} ({d.Budget})");
    PrintIndented(d.Left, depth + 1);            // the context changes on the way down
    PrintIndented(d.Right, depth + 1);
}

static int Total(Dept? d) => d is null ? 0 : d.Budget + Total(d.Left) + Total(d.Right);

// Does some root-to-leaf path have budgets that add up to target?
static bool HasPathTotal(Dept? d, int target)
{
    if (d is null) return false;
    int remaining = target - d.Budget;                        // passed down
    if (d.Left is null && d.Right is null) return remaining == 0;
    return HasPathTotal(d.Left, remaining) || HasPathTotal(d.Right, remaining);   // returned up
}

record Dept(string Name, int Budget, Dept? Left = null, Dept? Right = null);

Company (0)
  Sales (120)
    Inside (40)
    Field (65)
  Operations (90)
    Warehouse (30)
total budget = 345
path totaling 225? False
path totaling 200? False


The indented listing shows the depth parameter at work: each call prints with twice its depth in spaces and hands `depth + 1` to its children, so the context changes at every level without any global state. `Total` returns 345, combining each subtree's result on the way back up. `HasPathTotal` does both: the remaining target flows down, shrinking at each node, and a Boolean flows up, combined with `||` because one successful path is enough. The path Company, Sales, Field totals 0 + 120 + 65 = 185 and Company, Sales, Inside totals 160; Company, Operations, Warehouse totals 120, so neither 225 nor 200 is a path total and both print `False`. Change the target to 185 and the method returns `True` after exploring only the Sales branch. A version that tested `remaining == 0` at every node rather than at leaves would answer a different question, whether any prefix of a path hits the target.

```{index} Code Completion: Label Each Node with Its Depth
```

### Code Completion: Label Each Node with Its Depth

Complete `PrintDepths` so that it prints every node's value followed by its depth, visiting in preorder. The expected output is `8:0`, `3:1`, `1:2`, `6:2`, `10:1`, one per line. The depth is information that flows down, so think about what each recursive call must receive.

In [ ]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

PrintDepths(root, 0);

static void PrintDepths(TreeNode? node, int depth)
{
    if (node is null) return;
    Console.WriteLine($"{node.Value}:{depth}");
    // Your code starts here.
    PrintDepths(node.Left, depth);
    PrintDepths(node.Right, depth);
    // Your code ends here.
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }
    public TreeNode(int value) => Value = value;
}

In [ ]:
// Solution
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

PrintDepths(root, 0);

static void PrintDepths(TreeNode? node, int depth)
{
    if (node is null) return;
    Console.WriteLine($"{node.Value}:{depth}");
    PrintDepths(node.Left, depth + 1);    // a child is one level deeper than its parent
    PrintDepths(node.Right, depth + 1);
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }
    public TreeNode(int value) => Value = value;
}

8:0
3:1
1:2
6:2
10:1


The starter passes `depth` unchanged, so every node reports depth 0: the context was passed down but never adjusted. Adding one at each call is the whole fix. A tempting alternative, `depth++` before the calls, also works here but mutates the parameter, which becomes confusing when the method does more after the recursive calls; `depth + 1` leaves the current call's own depth intact.

```{index} Traversal Orders
```

## Traversal Orders




```{index} preorder traversal; inorder traversal; postorder traversal
```


A **traversal** visits every node in a tree. For binary trees, the common depth-first traversal orders differ only in when the current node is processed:

- **Preorder**: node, left, right
- **Inorder**: left, node, right
- **Postorder**: left, right, node


In [1]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

Console.Write("Preorder:  ");
Preorder(root);
Console.WriteLine();

Console.Write("Inorder:   ");
Inorder(root);
Console.WriteLine();

Console.Write("Postorder: ");
Postorder(root);
Console.WriteLine();

void Preorder(TreeNode? node)
{
    if (node is null) return;
    Console.Write(node.Value + " ");
    Preorder(node.Left);
    Preorder(node.Right);
}

void Inorder(TreeNode? node)
{
    if (node is null) return;
    Inorder(node.Left);
    Console.Write(node.Value + " ");
    Inorder(node.Right);
}

void Postorder(TreeNode? node)
{
    if (node is null) return;
    Postorder(node.Left);
    Postorder(node.Right);
    Console.Write(node.Value + " ");
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


Preorder:  8 3 1 6 10 
Inorder:   1 3 6 8 10 
Postorder: 1 6 3 10 8 


The tree shape is the same in all three traversals. Only the timing of `Console.Write` changes.


**Visualize it.** OpenDSA's [Binary Tree Traversals](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS2/html/BinaryTreeTraversal.html) slideshows animate preorder, inorder, and postorder visits on one tree, so the timing difference above is visible.

**Common mistake.** Moving the `Visit` call without moving the recursive calls with it. Printing before, between, or after the two child calls is the whole difference between preorder, inorder, and postorder; a misplaced print produces an order that is none of the three.


Trace the same five-node tree with the root at 8. In preorder, you process 8 before entering the call for 3. In inorder, the call at 8 stays suspended until the entire left subtree returns. In postorder, both subtrees finish before 8 is processed.

| Order | Sequence | Use in a hierarchy |
| --- | --- | --- |
| Preorder | 8, 3, 1, 6, 10 | Print a department heading before its teams. |
| Inorder | 1, 3, 6, 8, 10 | List keys in order when the tree is a BST. |
| Postorder | 1, 6, 3, 10, 8 | Total child budgets before computing a department total. |

Inorder traversal is sorted here because this particular tree obeys the binary search tree rule. Inorder traversal of an arbitrary binary tree is not necessarily sorted.

Every traversal visits each node exactly once. If processing a node costs constant time, total work is Θ(n). Recursive calls retain their local state while waiting for descendants, so the largest number of active calls is proportional to height: O(h + 1) extra space for a nonempty tree. Returning a list of all values also uses Θ(n) space for that result. The output list and the call stack are different costs.

**Predict:** move 6 from the right child of 3 to the left child of 10. Which traversal sequences change? The node count stays the same, but the order depends on the references, not only the stored values.

```{index} Code Tracing: Three Orders for a Changed Tree
```

### Code Tracing: Three Orders for a Changed Tree

Take the five-node tree from the example above and move node 6 from the right child of 3 to the left child of 10. Write the preorder, inorder, and postorder sequences for the new tree before running the solution, and say which of the three changed from the original and why.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Right.Left = new TreeNode(6);          // 6 moved under 10

Console.WriteLine("Preorder:  " + string.Join(" ", Walk(root, "pre")));
Console.WriteLine("Inorder:   " + string.Join(" ", Walk(root, "in")));
Console.WriteLine("Postorder: " + string.Join(" ", Walk(root, "post")));

static List<int> Walk(TreeNode? node, string order, List<int>? acc = null)
{
    acc ??= new List<int>();
    if (node is null) return acc;
    if (order == "pre") acc.Add(node.Value);
    Walk(node.Left, order, acc);
    if (order == "in") acc.Add(node.Value);
    Walk(node.Right, order, acc);
    if (order == "post") acc.Add(node.Value);
    return acc;
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }
    public TreeNode(int value) => Value = value;
}

Preorder:  8 3 1 10 6
Inorder:   1 3 8 6 10
Postorder: 1 3 6 10 8


Preorder becomes `8 3 1 10 6`, inorder `1 3 8 6 10`, and postorder `1 3 6 10 8`. All three changed, because the position of 6 relative to 3 and 10 changed. The inorder sequence is no longer sorted: 6 is now in the right subtree of 8 but smaller than 8, which breaks the binary search tree rule that the original tree happened to satisfy. Inorder traversal sorts the values only when the tree is a BST, a point the next section builds on.

```{index} Height and Shape
```

## Height and Shape




```{index} tree height; tree depth
```


A node's **depth** is its distance from the root. A tree's **height** is the longest downward path from the root to a leaf. Height matters because many tree operations follow one path from the root toward a leaf.


In [1]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);
root.Left.Right.Left = new TreeNode(4);

Console.WriteLine($"Height: {Height(root)}");

int Height(TreeNode? node)
{
    if (node is null)
    {
        return -1;
    }

    return 1 + Math.Max(Height(node.Left), Height(node.Right));
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


Height: 3


This version defines an empty tree as height `-1`, so a single-node tree has height `0`. Some texts define height differently; always check the convention being used.


**Common mistake.** Returning 0 for a leaf and `1 + max(children)` for an internal node but forgetting the `null` case, or returning 0 for `null` and 1 for a leaf. The two conventions differ by one at every level; pick one, state it, and test it on an empty tree and a single node.


Height counts edges in this chapter. Null has height -1, a leaf has height zero, and every internal node adds one to the larger child height. For the six-node example, the call at 6 returns one, the call at 3 returns two, and the root returns three. Depth belongs to a specific node; height summarizes a subtree.

Height computation visits both children, so it costs Θ(n), even when the tree is short. A search may follow only one path; computing the height of an uncached tree examines every node. In a chain of n nodes, height is n - 1 and recursion can require n active calls. Large adversarial chains can exhaust the call stack. Production trees often control height or use iterative algorithms.

**Check the boundaries:** an empty tree has zero nodes, zero leaves, and height -1. A singleton has one node, one leaf, and height zero. A chain with several nodes still has only one leaf. These cases distinguish correct base cases from methods that merely work on a symmetric example.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A method prints a department heading and then prints each of its sub-departments' headings beneath it. Which traversal order is that, and what changes if totals must be printed instead of headings?

```{dropdown} Answer
Preorder: the node is processed before its children. Totals depend on the children's totals, so they must be computed in postorder, after both subtrees have returned; the heading can be printed first and the total last, which is two visits of the same node at different moments.
```

2. Why does the maximum number of active calls during a traversal depend on height rather than on the number of nodes?

```{dropdown} Answer
A call stays active only while its descendants are being processed, so the active calls at any moment form one root-to-node path. The longest such path is the height, plus one for the root. A balanced tree of a million nodes needs about 20 active frames; a chain of the same size needs a million.
```

3. `Sum(node.Left); Sum(node.Right); return node.Value;` compiles and runs. What is wrong with it in information-flow terms?

```{dropdown} Answer
It is written as collect-and-return but discards what it collects: the two recursive results are never captured or added. Every call returns only its own value, so the method reports the root's value instead of the tree's sum.
```

4. Which flow does the BST validator of the next section use when it passes a lower and an upper bound to each recursive call?

```{dropdown} Answer
Passing down combined with a returned Boolean: the bounds are context inherited from ancestors and narrowed at each node, and each call returns whether its subtree respects them, combined with `&&`.
```

```{index} Exercise
```

## Exercise




```{index} tree traversal; exercise
```


Complete `CountLeaves` so it returns the number of nodes with no children. For the tree built in the cell the expected output is `3`. A `null` node contributes 0, a node with no children contributes 1, and any other node contributes the sum of its two subtrees.


In [ ]:
// Exercise: Count leaves
// Complete CountLeaves so the program prints 3 for this tree.

using System;

var root = new TreeNode(5);
root.Left = new TreeNode(2);
root.Right = new TreeNode(9);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(3);

Console.WriteLine(CountLeaves(root));

int CountLeaves(TreeNode? node)
{
    // Your code starts here.
    return 0; // TODO: replace with leaf counting.



    // Your code ends here.
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


In [1]:
// Solution
using System;

var root = new TreeNode(5);
root.Left = new TreeNode(2);
root.Right = new TreeNode(9);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(3);

Console.WriteLine(CountLeaves(root));

int CountLeaves(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    if (node.Left is null && node.Right is null)
    {
        return 1;
    }

    return CountLeaves(node.Left) + CountLeaves(node.Right);
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


3


The output is `3`: the leaves are 1, 3, and 9. Node 2 has two children, so it contributes the sum of its subtrees rather than counting itself, and the root likewise.


```{index} Summary
```

## Summary




```{index} Summary
```


Tree algorithms usually follow the data: handle the current node, then recur into child subtrees. Traversal order controls when the current node is processed.

```{rubric} Footnotes
```


[^1701_tree_structure-1]: This chapter uses binary trees so examples stay focused. General trees can have any number of children.


[^development-reading]: Further reading: [OpenDSA CS2, Binary Trees Chapter Introduction](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS2/html/BinaryTreeIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
